# 📻 Radio QC — dò quảng cáo trong file phát sóng

**Chế độ A – Đối soát:** tìm các spot quảng cáo có trong thư viện của bạn, báo giờ phát, số lần phát, phát đủ hay thiếu.
Notebook chạy trên Colab bằng tài khoản Google của bạn; file nằm trong Google Drive của bạn.

**Chạy:** bấm **Runtime → Run all** (Ctrl+F9).
- Colab cảnh báo notebook *không do Google viết* → bấm **Run anyway** (Vẫn chạy).
- Colab xin quyền vào Google Drive → chọn tài khoản của bạn → **Cho phép**.

**Lần đầu** chưa cần file nào: notebook tạo trong Drive thư mục `RadioQC/` rồi nhắc bước tiếp theo. Thư mục gồm:
- `1_ThuVienQC/`: file gốc các spot quảng cáo **của bạn**, dài ít nhất 5 giây. **Tên file = tên spot.**
- `2_FilePhatSong/`: file phát sóng cần dò. Tên file nên có ngày giờ bắt đầu, ví dụ `CAO DIEM CHIEU 25.9.2026 16H30.mp3`.
- `3_KetQua/`: báo cáo Excel, mỗi file phát sóng một báo cáo.

Bỏ file vào hai thư mục đầu rồi **Run all** lần nữa. File đã xử lý được bỏ qua ở lần sau; muốn dò lại tất cả
(ví dụ vừa thêm spot mới) thì tick **CHAY_LAI_TAT_CA**. Nghe lại một lần phát: ô **3️⃣**, nhập **STT** trong báo cáo.

**Cột "Tình trạng":** **Đủ** = phát trọn spot · **Phát thiếu** = bị cắt đầu/cuối ·
**Cần nghe lại** = chỉ khớp một phần, có thể là spot khác dùng chung nhạc hiệu · **Không kiểm được đủ** = spot nằm ở mép file ghi.

Notebook không bao giờ xoá hay di chuyển file của bạn. Hướng dẫn đầy đủ: https://github.com/nguyenthanhhai-rf/radio-qc-colab


In [ ]:
#@title ⚙️ Cấu hình
THU_MUC = "RadioQC"  #@param {type:"string"}
CHAY_LAI_TAT_CA = False  #@param {type:"boolean"}


In [ ]:
#@title 1️⃣ Cài đặt và kiểm tra (không cần sửa)
import pathlib
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av"], check=True)
_FILES = {'radioqc/__init__.py': '"""Radio QC: find radio advertisements in long broadcast recordings."""\n\n__version__ = "0.1.0"\n', 'radioqc/audio.py': '"""Decode audio files into mono float32 arrays (PyAV / FFmpeg)."""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport av\nimport numpy as np\n\nAUDIO_EXTENSIONS = {".mp3", ".mp2", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".wma", ".mp4"}\n\n\ndef is_audio_file(path: Path) -> bool:\n    return path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS and not path.name.startswith(".")\n\n\ndef load_audio(path: str | Path, sr: int, start_s: float = 0.0,\n               duration_s: float | None = None) -> np.ndarray:\n    """Decode `path` to mono float32 in [-1, 1] at `sr` Hz.\n\n    A window (`start_s`, `duration_s`) relies on container seeking, which is only\n    approximate for some formats: use windows for listening, never for measurements.\n    """\n    chunks: list[np.ndarray] = []\n    have, skip = 0, 0\n    need = None if duration_s is None else max(0, int(round(duration_s * sr)))\n    with av.open(str(path)) as container:\n        if not container.streams.audio:\n            raise ValueError(f"không có luồng âm thanh trong {Path(path).name}")\n        stream = container.streams.audio[0]\n        resampler = av.AudioResampler(format="s16", layout="mono", rate=sr)\n        origin: float | None = 0.0\n        if start_s > 0:\n            container.seek(int(max(0.0, start_s - 1.0) * 1_000_000))\n            origin = None\n        finished = True\n        for frame in container.decode(stream):\n            if origin is None:\n                origin = frame.time if frame.time is not None else max(0.0, start_s - 1.0)\n                skip = max(0, int(round((start_s - origin) * sr)))\n            for out in resampler.resample(frame):\n                a = out.to_ndarray().reshape(-1)\n                chunks.append(a)\n                have += a.size\n            if need is not None and have >= skip + need:\n                finished = False\n                break\n        if finished:\n            for out in resampler.resample(None):\n                a = out.to_ndarray().reshape(-1)\n                chunks.append(a)\n                have += a.size\n    x = np.concatenate(chunks) if chunks else np.zeros(0, np.int16)\n    x = x[skip:] if need is None else x[skip:skip + need]\n    return x.astype(np.float32) / 32768.0\n', 'radioqc/fingerprint.py': '"""Landmark fingerprints: pairs of spectral peaks, robust to broadcast processing.\n\nEvery parameter below changes the hashes. Bump FP_VERSION whenever one changes so\nthat cached library fingerprints are rebuilt.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\nfrom numpy.lib.stride_tricks import sliding_window_view\nfrom scipy.ndimage import maximum_filter\n\nFP_VERSION = 1\nSR = 8000                 # fingerprint sample rate (Hz)\nN_FFT = 512               # 64 ms window\nHOP = 256                 # 32 ms hop\nFPS = SR / HOP            # frames per second (31.25)\nF_LO, F_HI = 6, 240       # bins used for peaks (~94 Hz - 3.75 kHz)\nPEAK_SIZE = 15            # local-maximum neighbourhood, in bins and frames\nPEAK_ABOVE_MEDIAN = 2.0   # log-magnitude above the frame median (~17 dB)\nLOUD_FLOOR = float(np.log(0.05))  # quieter frames carry no peaks\nFAN_OUT = 5               # targets paired with each anchor peak\nDT_MAX = 32               # target at most 32 frames (~1 s) after the anchor\nDF_MAX = 32               # target within +/-32 bins of the anchor\nPAIR_SCAN = 80            # later peaks scanned per anchor when pairing\nBLOCK = 8192              # frames per spectrogram block (~4.4 min), bounds memory\n\n\n@dataclass\nclass Fingerprint:\n    hashes: np.ndarray    # uint32, one per landmark\n    times: np.ndarray     # int32 anchor frame of each landmark\n    n_frames: int\n    duration_s: float\n\n\ndef _peaks(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, int]:\n    """Spectral peaks as (frame, bin) arrays sorted by frame, then bin."""\n    n_frames = 1 + (x.size - N_FFT) // HOP if x.size >= N_FFT else 0\n    if n_frames == 0:\n        empty = np.zeros(0, np.int64)\n        return empty, empty, 0\n    frames = sliding_window_view(x, N_FFT)[::HOP]\n    win = np.hanning(N_FFT).astype(np.float32)\n    pad = PEAK_SIZE // 2\n    ts, fs = [], []\n    for b0 in range(0, n_frames, BLOCK):\n        b1 = min(n_frames, b0 + BLOCK)\n        c0, c1 = max(0, b0 - pad), min(n_frames, b1 + pad)\n        spec = np.abs(np.fft.rfft(frames[c0:c1] * win, axis=1)).T\n        S = np.log(spec + 1e-6).astype(np.float32)\n        band = S[F_LO:F_HI]\n        is_max = band == maximum_filter(band, size=PEAK_SIZE, mode="constant", cval=-np.inf)\n        strong = band > np.median(band, axis=0, keepdims=True) + PEAK_ABOVE_MEDIAN\n        loud = S.max(axis=0) > LOUD_FLOOR\n        f, t = np.nonzero(is_max & strong & loud[None, :])\n        t = t + c0\n        keep = (t >= b0) & (t < b1)\n        ts.append(t[keep])\n        fs.append(f[keep] + F_LO)\n    t = np.concatenate(ts).astype(np.int64)\n    f = np.concatenate(fs).astype(np.int64)\n    order = np.lexsort((f, t))\n    return t[order], f[order], n_frames\n\n\ndef _pairs(t: np.ndarray, f: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    """Pair each anchor peak with up to FAN_OUT later peaks in its target zone."""\n    n = t.size\n    used = np.zeros(n, np.int32)\n    hashes, times = [], []\n    for s in range(1, min(PAIR_SCAN, n - 1) + 1):\n        i = np.arange(n - s)\n        j = i + s\n        dt = t[j] - t[i]\n        if dt.min() > DT_MAX:\n            break\n        df = f[j] - f[i]\n        ok = (dt >= 1) & (dt <= DT_MAX) & (np.abs(df) <= DF_MAX) & (used[i] < FAN_OUT)\n        i = i[ok]\n        used[i] += 1\n        hashes.append((f[i] << 13) | ((df[ok] + DF_MAX) << 6) | dt[ok])\n        times.append(t[i])\n    if not hashes:\n        return np.zeros(0, np.uint32), np.zeros(0, np.int32)\n    return np.concatenate(hashes).astype(np.uint32), np.concatenate(times).astype(np.int32)\n\n\ndef fingerprint(x: np.ndarray) -> Fingerprint:\n    """Fingerprint mono audio sampled at SR."""\n    x = np.ascontiguousarray(x, dtype=np.float32)\n    t, f, n_frames = _peaks(x)\n    hashes, times = _pairs(t, f)\n    return Fingerprint(hashes, times, n_frames, x.size / SR)\n', 'radioqc/library.py': '"""Spot library: fingerprints of known ads, indexed by hash and cached on disk."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom collections.abc import Callable\n\nimport numpy as np\n\nfrom .audio import is_audio_file, load_audio\nfrom .fingerprint import FP_VERSION, FPS, SR, Fingerprint, fingerprint\n\nMIN_SPOT_S = 5.0  # shorter clips (sound logos) cannot be matched reliably\n\n\n@dataclass\nclass Spot:\n    name: str\n    file: str\n    duration_s: float\n    hashes: np.ndarray        # sorted\n    times: np.ndarray         # aligned with hashes\n    content_secs: np.ndarray  # sorted whole seconds that contain landmarks\n\n    @property\n    def n_landmarks(self) -> int:\n        return int(self.hashes.size)\n\n\n@dataclass\nclass LibraryIndex:\n    spots: list[Spot]\n    hashes: np.ndarray        # every landmark of every spot, sorted by hash\n    spot_ids: np.ndarray\n    times: np.ndarray\n    problems: list[tuple[str, str]] = field(default_factory=list)\n\n    @classmethod\n    def from_fingerprints(cls, items: list[tuple[str, str, Fingerprint]],\n                          problems: list[tuple[str, str]] | None = None) -> LibraryIndex:\n        """items: (name, file, fingerprint) for each spot."""\n        spots = []\n        for name, file, fp in items:\n            o = np.argsort(fp.hashes, kind="stable")\n            secs = np.unique((fp.times / FPS).astype(np.int64))\n            spots.append(Spot(name, file, fp.duration_s, fp.hashes[o], fp.times[o], secs))\n        if spots:\n            h = np.concatenate([s.hashes for s in spots])\n            ids = np.concatenate([np.full(s.n_landmarks, k, np.int32) for k, s in enumerate(spots)])\n            t = np.concatenate([s.times for s in spots])\n            o = np.argsort(h, kind="stable")\n            h, ids, t = h[o], ids[o], t[o]\n        else:\n            h, ids, t = np.zeros(0, np.uint32), np.zeros(0, np.int32), np.zeros(0, np.int32)\n        return cls(spots, h, ids, t, list(problems or []))\n\n\ndef _meta(rel: str, st: os.stat_result) -> dict:\n    return {"file": rel, "size": st.st_size, "mtime_ns": st.st_mtime_ns, "fp_version": FP_VERSION}\n\n\ndef _load_cached(path: Path, meta: dict) -> Fingerprint | None:\n    try:\n        with np.load(path, allow_pickle=False) as z:\n            if json.loads(str(z["meta"])) != meta:\n                return None\n            return Fingerprint(z["hashes"], z["times"], int(z["n_frames"]), float(z["duration_s"]))\n    except (OSError, KeyError, ValueError):\n        return None\n\n\ndef _save_cached(path: Path, meta: dict, fp: Fingerprint) -> None:\n    tmp = path.with_name(path.name + ".tmp.npz")\n    np.savez(tmp, hashes=fp.hashes, times=fp.times, n_frames=fp.n_frames,\n             duration_s=fp.duration_s, meta=json.dumps(meta, sort_keys=True))\n    os.replace(tmp, path)\n\n\ndef build_library(folder: Path, cache_dir: Path, log: Callable[[str], None] = print) -> LibraryIndex:\n    """Fingerprint every audio file under `folder` (re-using the cache) and index them."""\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    files = sorted((p for p in folder.rglob("*") if is_audio_file(p)),\n                   key=lambda p: p.relative_to(folder).as_posix().lower())\n    items, problems = [], []\n    for p in files:\n        rel = p.relative_to(folder).as_posix()\n        meta = _meta(rel, p.stat())\n        cpath = cache_dir / (hashlib.sha1(rel.encode("utf-8")).hexdigest()[:16] + ".npz")\n        fp = _load_cached(cpath, meta)\n        if fp is None:\n            try:\n                fp = fingerprint(load_audio(p, SR))\n            except Exception as e:  # noqa: BLE001 - list every unreadable file, keep going\n                problems.append((rel, f"không đọc được file ({e})"))\n                continue\n            _save_cached(cpath, meta, fp)\n            log(f"  + vân tay: {rel}")\n        if fp.duration_s < MIN_SPOT_S:\n            problems.append((rel, f"quá ngắn ({fp.duration_s:.1f} s < {MIN_SPOT_S:.0f} s) để dò tin cậy"))\n        elif fp.hashes.size == 0:\n            problems.append((rel, "không có nội dung để lấy vân tay (im lặng?)"))\n        else:\n            items.append((p.stem, rel, fp))\n    stems = [n for n, _, _ in items]\n    items = [(rel.rsplit(".", 1)[0] if stems.count(n) > 1 else n, rel, fp) for n, rel, fp in items]\n    return LibraryIndex.from_fingerprints(items, problems)\n', 'radioqc/matcher.py': '"""Locate airings of library spots inside a broadcast fingerprint.\n\nAn airing shows up as many landmark pairs agreeing on one (spot, time offset).\nThresholds are fixed by tests (synthetic + the real 25/9 broadcast), not by feel.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom .fingerprint import DT_MAX, FPS, Fingerprint\nfrom .library import LibraryIndex, Spot\n\nMIN_PAIRS_ABS = 10        # consistent landmark pairs needed to consider an airing\n                          # (real 25/9 broadcast: random offsets reach 2-6 pairs; a worst-aligned\n                          # re-airing kept only 8.5% of pairs, so 20 would miss 5 s spots)\nMIN_PAIRS_REL = 0.02      # ... or this share of the spot\'s landmarks, if larger\nMIN_COVERAGE = 0.4        # share of the spot\'s content seconds that must be found\nMIN_MATCHED_S = 4         # distinct matched seconds\nPARTIAL_GAP_S = 2.0       # missing head/tail longer than this => partial airing\nMID_GAP_S = 4             # unmatched content seconds in a row inside the airing => doubtful\nFULL_INNER_COVERAGE = 0.8  # below this share inside the matched span => doubtful\nTIE_S = 3.0               # matched durations this close are decided by coverage\nQUERY_BLOCK = 200_000     # query landmarks per block when collecting candidates\n\n\n@dataclass\nclass Airing:\n    spot_id: int\n    name: str\n    offset_s: float           # broadcast time aligned with the spot\'s first sample\n    start_s: float\n    end_s: float\n    coverage: float           # share of the spot\'s content seconds found\n    matched_s: int            # distinct spot seconds found\n    missing_head_s: float\n    missing_tail_s: float\n    cut_by_file_start: bool   # the missing head lies before the recording started\n    cut_by_file_end: bool     # the missing tail lies after the recording ended\n    n_pairs: int\n    inner_coverage: float = 1.0  # share of content seconds found between the first and last match\n    missing_mid_s: int = 0       # longest run of unmatched content seconds inside the airing\n    also_matched: list[str] = field(default_factory=list)\n\n    @property\n    def aired_s(self) -> float:\n        return self.end_s - self.start_s\n\n    @property\n    def status(self) -> str:\n        """\'Đủ\', or why not: doubts about identity outrank a short airing."""\n        doubt, short, cut = [], [], []\n        if self.missing_mid_s > MID_GAP_S:\n            doubt.append(f"không khớp đoạn giữa ~{self.missing_mid_s} s (có thể là spot khác cùng nhạc hiệu)")\n        if self.inner_coverage < FULL_INNER_COVERAGE:\n            doubt.append(f"chỉ khớp {100 * self.inner_coverage:.0f}% phần đã phát")\n        if self.missing_head_s > PARTIAL_GAP_S:\n            if self.cut_by_file_start:\n                cut.append("đầu nằm ngoài file ghi")\n            else:\n                short.append(f"thiếu đầu ~{self.missing_head_s:.0f} s")\n        if self.missing_tail_s > PARTIAL_GAP_S:\n            if self.cut_by_file_end:\n                cut.append("cuối nằm ngoài file ghi")\n            else:\n                short.append(f"thiếu cuối ~{self.missing_tail_s:.0f} s")\n        if doubt:\n            return "Cần nghe lại: " + ", ".join(doubt + short + cut)\n        if short:\n            return "Phát thiếu: " + ", ".join(short + cut)\n        if cut:\n            return "Không kiểm được đủ: " + ", ".join(cut)\n        return "Đủ"\n\n\ndef find_airings(index: LibraryIndex, query: Fingerprint) -> list[Airing]:\n    """All airings of library spots in `query`, sorted by start time."""\n    if not index.spots or query.hashes.size == 0:\n        return []\n    order = np.argsort(query.times, kind="stable")\n    qh, qt = query.hashes[order], query.times[order].astype(np.int64)\n    found = []\n    for sid, off in _candidates(index, qh, qt, query.n_frames):\n        a = _measure(index.spots[sid], sid, off, qh, qt, query.duration_s)\n        if a is not None:\n            found.append(a)\n    return sorted(_resolve_overlaps(found), key=lambda a: a.start_s)\n\n\ndef _expand(sorted_hashes: np.ndarray, h: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    """Index pairs (into h, into sorted_hashes) of all equal hashes."""\n    lo = np.searchsorted(sorted_hashes, h, "left")\n    cnt = np.searchsorted(sorted_hashes, h, "right") - lo\n    total = int(cnt.sum())\n    qi = np.repeat(np.arange(h.size), cnt)\n    li = np.repeat(lo, cnt) + np.arange(total) - np.repeat(np.cumsum(cnt) - cnt, cnt)\n    return qi, li\n\n\ndef _neighbour(k: np.ndarray, c: np.ndarray, step: int) -> np.ndarray:\n    """Count stored at key k+step (0 where absent); `k` is sorted and unique."""\n    pos = np.searchsorted(k, k + step)\n    hit = pos < k.size\n    hit[hit] &= k[pos[hit]] == (k + step)[hit]\n    out = np.zeros_like(c)\n    out[hit] = c[pos[hit]]\n    return out\n\n\ndef _candidates(index: LibraryIndex, qh: np.ndarray, qt: np.ndarray,\n                n_frames: int) -> list[tuple[int, int]]:\n    """(spot_id, offset in frames) of dense offset peaks, strongest first."""\n    longest = max(int(np.ceil(s.duration_s * FPS)) for s in index.spots) + DT_MAX + 2\n    width = n_frames + 2 * longest + 2\n    keys, counts = [], []\n    for b0 in range(0, qh.size, QUERY_BLOCK):\n        qi, li = _expand(index.hashes, qh[b0:b0 + QUERY_BLOCK])\n        if qi.size == 0:\n            continue\n        off = qt[b0:b0 + QUERY_BLOCK][qi] - index.times[li].astype(np.int64)\n        key = index.spot_ids[li].astype(np.int64) * width + off + longest\n        k, c = np.unique(key, return_counts=True)\n        keys.append(k[c >= 2])\n        counts.append(c[c >= 2])\n    if not keys:\n        return []\n    k, inv = np.unique(np.concatenate(keys), return_inverse=True)\n    c = np.bincount(inv, weights=np.concatenate(counts)).astype(np.int64)\n    c3 = c + _neighbour(k, c, -1) + _neighbour(k, c, +1)   # tolerate +/-1 frame of jitter\n    sid, off = k // width, k % width - longest\n    need = np.array([max(MIN_PAIRS_ABS, MIN_PAIRS_REL * s.n_landmarks) for s in index.spots])\n    ok = np.flatnonzero(c3 >= need[sid])\n    ok = ok[np.argsort(-c3[ok], kind="stable")]\n    taken: dict[int, list[int]] = {}\n    out = []\n    for s, o in zip(sid[ok].tolist(), off[ok].tolist(), strict=True):\n        window = max(1, int(0.9 * index.spots[s].duration_s * FPS))\n        if any(abs(o - p) < window for p in taken.get(s, [])):\n            continue\n        taken.setdefault(s, []).append(o)\n        out.append((s, o))\n    return out\n\n\ndef _measure(spot: Spot, sid: int, off: int, qh: np.ndarray, qt: np.ndarray,\n             file_s: float) -> Airing | None:\n    """Coverage and boundaries of one candidate airing, or None if too weak."""\n    n_sf = int(np.ceil(spot.duration_s * FPS))\n    a, b = np.searchsorted(qt, [off - 1, off + n_sf + 2])\n    qi, li = _expand(spot.hashes, qh[a:b])\n    if qi.size == 0:\n        return None\n    tref = spot.times[li].astype(np.int64)\n    tref = tref[np.abs(qt[a:b][qi] - tref - off) <= 1]\n    secs = np.unique((tref / FPS).astype(np.int64))\n    if secs.size < MIN_MATCHED_S:\n        return None\n    content = spot.content_secs\n    coverage = secs.size / content.size\n    if coverage < MIN_COVERAGE:\n        return None\n    missing_head = float(secs[0] - content[0])\n    missing_tail = float(content[-1] - secs[-1])\n    inside = content[(content >= secs[0]) & (content <= secs[-1])]\n    offset_s = off / FPS\n    start_s = offset_s + (tref.min() / FPS if missing_head > PARTIAL_GAP_S else 0.0)\n    end_s = offset_s + (min(spot.duration_s, tref.max() / FPS + 1.0) if missing_tail > PARTIAL_GAP_S\n                        else spot.duration_s)\n    return Airing(\n        spot_id=sid, name=spot.name, offset_s=offset_s,\n        start_s=max(0.0, start_s), end_s=min(file_s, end_s),\n        coverage=float(coverage), matched_s=int(secs.size),\n        missing_head_s=missing_head, missing_tail_s=missing_tail,\n        cut_by_file_start=offset_s + secs[0] < PARTIAL_GAP_S,\n        cut_by_file_end=offset_s + secs[-1] + 1.0 > file_s - PARTIAL_GAP_S,\n        n_pairs=int(tref.size),\n        inner_coverage=float(secs.size / inside.size),\n        missing_mid_s=_longest_run(~np.isin(inside, secs)))\n\n\ndef _longest_run(flags: np.ndarray) -> int:\n    best = run = 0\n    for f in flags.tolist():\n        run = run + 1 if f else 0\n        best = max(best, run)\n    return best\n\n\ndef _resolve_overlaps(airings: list[Airing]) -> list[Airing]:\n    """Keep the spot that explains the longest stretch; list the others as \'also matched\'.\n\n    Stretches within ~TIE_S of each other are decided by coverage, so a complete 15 s\n    cut-down beats the 30 s version that only matched those same 15 s, while a 30 s\n    version that explains 18 s beats the 15 s cut-down inside it.\n    """\n    def score(a: Airing) -> float:\n        return a.aired_s + TIE_S * a.coverage\n\n    kept: list[Airing] = []\n    for a in sorted(airings, key=lambda a: (-score(a), a.name)):\n        host = next((k for k in kept\n                     if min(a.end_s, k.end_s) - max(a.start_s, k.start_s) > 0.5 * max(a.aired_s, 1e-6)),\n                    None)\n        if host is None:\n            kept.append(a)\n        elif a.name not in host.also_matched:\n            host.also_matched.append(a.name)\n    return kept\n', 'radioqc/naming.py': '"""Program name and start time from a broadcast file name.\n\nExample: \'CAO DIEM CHIEU 25.9.2026 16H30.mp3\' -> CAO DIEM CHIEU, 2026-09-25 16:30.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass\nfrom datetime import date, datetime, time\nfrom pathlib import Path\n\n_DMY = re.compile(r"(?<!\\d)(\\d{1,2})[.\\-_/](\\d{1,2})[.\\-_/](\\d{4})(?!\\d)")\n_YMD = re.compile(r"(?<!\\d)(\\d{4})[.\\-_/](\\d{1,2})[.\\-_/](\\d{1,2})(?!\\d)")\n# After the date any separator is a time (\'16H30\', \'16:30\', \'16.30\'); before the date only an\n# explicit hour mark counts, because \'91.00\' / \'22.00\' there are usually part of the program name.\n_HM_AFTER = re.compile(r"(?<!\\d)(\\d{1,2})\\s*(?:[hHgG:.]|giờ)\\s*(\\d{2})(?!\\d)")\n_HM_BEFORE = re.compile(r"(?<!\\d)(\\d{1,2})\\s*(?:[hHgG]|giờ)\\s*(\\d{2})(?!\\d)")\n\n\n@dataclass\nclass BroadcastName:\n    program: str\n    day: date | None\n    start: datetime | None\n\n\ndef _find_day(stem: str) -> tuple[date | None, tuple[int, int] | None]:\n    for rx, day_first in ((_DMY, True), (_YMD, False)):\n        for m in rx.finditer(stem):\n            a, b, c = (int(g) for g in m.groups())\n            d, mo, y = (a, b, c) if day_first else (c, b, a)\n            try:\n                return date(y, mo, d), m.span()\n            except ValueError:\n                continue\n    return None, None\n\n\ndef parse_broadcast_name(filename: str) -> BroadcastName:\n    stem = Path(filename).stem\n    day, span = _find_day(stem)\n    if day is None or span is None:\n        return BroadcastName(stem.strip(), None, None)\n    parts = [stem[:span[0]], stem[span[1]:]]\n    start = None\n    # the first valid time after the date, else the valid time closest before it\n    for i, rx, order in ((1, _HM_AFTER, 1), (0, _HM_BEFORE, -1)):\n        for m in list(rx.finditer(parts[i]))[::order]:\n            hh, mm = int(m.group(1)), int(m.group(2))\n            if hh < 24 and mm < 60:\n                start = datetime.combine(day, time(hh, mm))\n                parts[i] = parts[i][:m.start()] + " " + parts[i][m.end():]\n                break\n        if start:\n            break\n    program = re.sub(r"[\\s_]+", " ", " ".join(parts)).strip(" -._")\n    return BroadcastName(program or stem.strip(), day, start)\n', 'radioqc/pipeline.py': '"""Batch processing of broadcast files on the Google Drive layout (mode A)."""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport time\nimport traceback\nfrom dataclasses import dataclass\nfrom datetime import datetime\nfrom pathlib import Path\nfrom collections.abc import Callable\n\nimport numpy as np\n\nfrom . import __version__\nfrom .audio import is_audio_file, load_audio\nfrom .fingerprint import FP_VERSION, SR, fingerprint\nfrom .library import LibraryIndex, build_library\nfrom .matcher import find_airings\nfrom .naming import parse_broadcast_name\nfrom .report import clock, details_rows, hms, summary_rows, write_report\n\nLog = Callable[[str], None]\n\n\n@dataclass\nclass Layout:\n    """Folders under the Drive root, e.g. /content/drive/MyDrive/RadioQC."""\n    root: Path\n\n    @property\n    def library(self) -> Path:\n        return self.root / "1_ThuVienQC"\n\n    @property\n    def inbox(self) -> Path:\n        return self.root / "2_FilePhatSong"\n\n    @property\n    def results(self) -> Path:\n        return self.root / "3_KetQua"\n\n    @property\n    def cache(self) -> Path:\n        return self.root / ".cache"\n\n    def ensure(self) -> None:\n        for p in (self.library, self.inbox, self.results, self.cache):\n            p.mkdir(parents=True, exist_ok=True)\n\n\n@dataclass\nclass FileResult:\n    file: str\n    status: str\n    airings: int = 0\n    report: str = ""\n    seconds: float = 0.0\n\n\ndef broadcast_files(layout: Layout) -> list[Path]:\n    if not layout.inbox.is_dir():\n        return []\n    return sorted((p for p in layout.inbox.iterdir() if is_audio_file(p)), key=lambda p: p.name.lower())\n\n\ndef _write_json(path: Path, data) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=1), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef load_registry(layout: Layout) -> dict:\n    try:\n        return json.loads((layout.cache / "processed.json").read_text(encoding="utf-8"))\n    except (FileNotFoundError, json.JSONDecodeError):\n        return {}\n\n\ndef _stamp(p: Path) -> dict:\n    st = p.stat()\n    return {"size": st.st_size, "mtime_ns": st.st_mtime_ns}\n\n\ndef is_done(layout: Layout, registry: dict, p: Path) -> bool:\n    """Processed before, unchanged since, and its report still exists."""\n    r = registry.get(p.name)\n    if not r or {k: r.get(k) for k in ("size", "mtime_ns")} != _stamp(p):\n        return False\n    report = r.get("report", "")\n    return bool(report) and (layout.results / report).is_file()\n\n\ndef _has_spot_files(layout: Layout) -> bool:\n    return layout.library.is_dir() and any(is_audio_file(p) for p in layout.library.rglob("*"))\n\n\ndef self_check(layout: Layout) -> list[tuple[str, str, str]]:\n    """Rows of (item, \'OK\' | \'CHỜ FILE\' | \'CẢNH BÁO\' | \'LỖI\', detail) shown before running.\n\n    Empty folders are a normal state (each user adds their own files, if they have any),\n    so they read \'CHỜ FILE\'; \'LỖI\' is kept for real problems such as unreadable files.\n    """\n    rows = [("Thư mục gốc", "OK" if layout.root.is_dir() else "LỖI", str(layout.root))]\n    if _has_spot_files(layout):  # decode every spot now (cached), so a useless library shows up before running\n        index = build_library(layout.library, layout.cache / "library", log=lambda msg: None)\n        bad = "; ".join(f"{rel}: {why}" for rel, why in index.problems)\n        state = "LỖI" if not index.spots else ("CẢNH BÁO" if bad else "OK")\n        detail = f"{len(index.spots)} spot dùng được" + (f"; không dùng được: {bad}" if bad else "")\n    else:\n        state, detail = "CHỜ FILE", "chưa có file spot: bỏ file spot quảng cáo của bạn vào đây (tên file = tên spot)"\n    rows.append(("Thư viện QC (1_ThuVienQC)", state, detail))\n    files = broadcast_files(layout)\n    if not files:\n        rows.append(("File phát sóng (2_FilePhatSong)", "CHỜ FILE", "chưa có file: bỏ file phát sóng cần dò vào đây"))\n        return rows\n    registry = load_registry(layout)\n    todo = sum(not is_done(layout, registry, p) for p in files)\n    rows.append(("File phát sóng (2_FilePhatSong)", "OK", f"{len(files)} file, {todo} file chưa xử lý"))\n    no_time = [p.name for p in files if parse_broadcast_name(p.name).start is None]\n    rows.append(("Tên file có ngày giờ", "CẢNH BÁO" if no_time else "OK",\n                 "không đọc được giờ bắt đầu (báo cáo sẽ dùng giờ tương đối): " + ", ".join(no_time)\n                 if no_time else "đọc được giờ bắt đầu của mọi file"))\n    return rows\n\n\ndef next_step(layout: Layout) -> str | None:\n    """What the user should do next while a folder is still empty; None once both have files."""\n    no_spots, no_files = not _has_spot_files(layout), not broadcast_files(layout)\n    if no_spots and no_files:\n        return ("✅ Thư mục RadioQC đã sẵn sàng trong Google Drive của bạn. Bước tiếp theo: bỏ file spot quảng cáo "\n                "của bạn vào 1_ThuVienQC và file phát sóng cần dò vào 2_FilePhatSong, rồi Run all lần nữa.")\n    if no_spots:\n        return ("Bước tiếp theo: bỏ file spot quảng cáo của bạn vào 1_ThuVienQC (tên file = tên spot), "\n                "rồi Run all lần nữa.")\n    if no_files:\n        return "Bước tiếp theo: bỏ file phát sóng cần dò vào 2_FilePhatSong, rồi Run all lần nữa."\n    return None\n\n\ndef run_mode_a(layout: Layout, rerun: bool = False, log: Log = print) -> list[FileResult]:\n    """Process every new broadcast file; one failing file never stops the batch."""\n    layout.ensure()\n    t0 = time.time()\n    index = build_library(layout.library, layout.cache / "library", log=log)\n    for rel, why in index.problems:\n        log(f"⚠️ Thư viện: {rel}: {why}")\n    if not index.spots:\n        if index.problems:\n            log("❌ Không có file spot nào dùng được trong 1_ThuVienQC (xem cảnh báo ở trên): dừng, không tạo báo cáo.")\n        else:\n            log("ℹ️ Chưa có file spot trong 1_ThuVienQC nên chưa có gì để dò. Bỏ file spot vào rồi chạy lại.")\n        return []\n    files = broadcast_files(layout)\n    if not files:\n        log("ℹ️ Chưa có file phát sóng trong 2_FilePhatSong. Bỏ file vào rồi chạy lại.")\n        return []\n    log(f"Thư viện: {len(index.spots)} spot ({time.time() - t0:.0f} s)")\n    registry = load_registry(layout)\n    results = []\n    for p in files:\n        if not rerun and is_done(layout, registry, p):\n            r = registry[p.name]\n            results.append(FileResult(p.name, "Bỏ qua (đã xử lý)", r.get("airings", 0), r["report"]))\n            continue\n        t1 = time.time()\n        out = _out_stem(p, files)\n        try:\n            n = _process(layout, index, p, out, log)\n        except Exception as e:  # noqa: BLE001 - report it and continue with the next file\n            log(f"❌ {p.name}: {e}\\n{traceback.format_exc()}")\n            results.append(FileResult(p.name, f"LỖI: {e}", seconds=time.time() - t1))\n            continue\n        registry[p.name] = {**_stamp(p), "report": out + ".xlsx", "airings_file": out + ".json",\n                            "airings": n, "processed_at": datetime.now().isoformat(timespec="seconds")}\n        _write_json(layout.cache / "processed.json", registry)\n        results.append(FileResult(p.name, "OK", n, out + ".xlsx", time.time() - t1))\n    return results\n\n\ndef _out_stem(p: Path, files: list[Path]) -> str:\n    """Output name for `p`; \'X.mp3\' and \'X.wav\' get \'X (mp3)\' and \'X (wav)\' so they never collide."""\n    twins = sum(f.stem.lower() == p.stem.lower() for f in files)\n    return p.stem if twins < 2 else f"{p.stem} ({p.suffix.lstrip(\'.\').lower()})"\n\n\ndef _process(layout: Layout, index: LibraryIndex, p: Path, out: str, log: Log) -> int:\n    name = parse_broadcast_name(p.name)\n    x = load_audio(p, SR)\n    duration = x.size / SR\n    fp = fingerprint(x)\n    del x\n    airings = find_airings(index, fp)\n    info = [\n        ("File", p.name),\n        ("Chương trình", name.program),\n        ("Bắt đầu (đọc từ tên file)", name.start.strftime("%d/%m/%Y %H:%M") if name.start\n         else "KHÔNG ĐỌC ĐƯỢC: giờ trong báo cáo là giờ tương đối tính từ đầu file"),\n        ("Thời lượng file", hms(duration)),\n        ("Số spot trong thư viện", str(len(index.spots))),\n        ("Số lần phát tìm thấy", str(len(airings))),\n        ("Chế độ", "A – Đối soát"),\n        ("Phiên bản", f"radioqc {__version__}, vân tay v{FP_VERSION}"),\n        ("Xử lý lúc", datetime.now().strftime("%d/%m/%Y %H:%M:%S")),\n    ] + [("Cảnh báo thư viện", f"{rel}: {why}") for rel, why in index.problems]\n    write_report(layout.results / (out + ".xlsx"), details_rows(airings, index, name),\n                 summary_rows(airings, index, name), info)\n    _write_json(layout.cache / "airings" / (out + ".json"),\n                [{"stt": k, "name": a.name, "start_s": a.start_s, "end_s": a.end_s,\n                  "clock": clock(name, a.start_s)[1]} for k, a in enumerate(airings, 1)])\n    log(f"✅ {p.name}: dài {hms(duration)}, {len(airings)} lần phát → 3_KetQua/{out}.xlsx")\n    return len(airings)\n\n\ndef listen_clip(layout: Layout, file: str = "", stt: int = 1, pad_s: float = 1.0,\n                sr: int = 16000) -> tuple[np.ndarray, int, str]:\n    """Audio of airing `stt` in `file` (default: the most recently processed file)."""\n    registry = load_registry(layout)\n    if not registry:\n        raise ValueError("Chưa có lần phát nào để nghe: bỏ file vào hai thư mục rồi chạy ô 2️⃣ trước.")\n    if not file:\n        file = max(registry, key=lambda k: registry[k].get("processed_at", ""))\n    file = file.strip()\n    if file not in registry:  # accept the name without its extension too\n        same = [k for k in registry if Path(k).stem == file]\n        if len(same) > 1:\n            raise ValueError(f"Có nhiều file tên \'{file}\': {\', \'.join(sorted(same))}. Hãy gõ cả phần đuôi.")\n        file = same[0] if same else file\n    if file not in registry:\n        raise ValueError(f"Không thấy \'{file}\'. Các file đã xử lý: {\', \'.join(sorted(registry))}")\n    sidecar = registry[file].get("airings_file") or Path(file).stem + ".json"\n    rows = json.loads((layout.cache / "airings" / sidecar).read_text(encoding="utf-8"))\n    if not 1 <= stt <= len(rows):\n        raise ValueError(f"\'{file}\' có {len(rows)} lần phát: STT phải từ 1 đến {len(rows)}.")\n    if not (layout.inbox / file).is_file():\n        raise ValueError(f"File \'{file}\' không còn trong 2_FilePhatSong nên không nghe lại được.")\n    r = rows[stt - 1]\n    start = max(0.0, r["start_s"] - pad_s)\n    x = load_audio(layout.inbox / file, sr, start_s=start, duration_s=r["end_s"] + pad_s - start)\n    return x, sr, f"{file}: STT {stt} — {r[\'name\']} lúc {r[\'clock\']}"\n', 'radioqc/report.py': '"""Excel report for one broadcast file (openpyxl only: pure Python, no compiled deps)."""\nfrom __future__ import annotations\n\nimport os\nfrom datetime import timedelta\nfrom pathlib import Path\n\nfrom openpyxl import Workbook\nfrom openpyxl.styles import Font\n\nfrom .library import LibraryIndex\nfrom .matcher import Airing\nfrom .naming import BroadcastName\n\nDETAIL_COLUMNS = ["STT", "Ngày", "Giờ bắt đầu", "Giờ kết thúc", "Thời lượng phát (s)", "Tên spot",\n                  "Thời lượng spot (s)", "Mức khớp (%)", "Tình trạng", "Cũng khớp", "Vị trí trong file"]\nSUMMARY_COLUMNS = ["Tên spot", "Số lần phát", "Các giờ phát", "Số lần phát thiếu", "Số lần cần nghe lại"]\n\n\ndef hms(seconds: float) -> str:\n    s = int(round(seconds))\n    return f"{s // 3600:02d}:{s % 3600 // 60:02d}:{s % 60:02d}"\n\n\ndef clock(name: BroadcastName, seconds: float) -> tuple[str, str]:\n    """(date, time) of a position in the file; \'+hh:mm:ss\' when the start time is unknown."""\n    if name.start is None:\n        return (name.day.strftime("%d/%m/%Y") if name.day else ""), "+" + hms(seconds)\n    t = name.start + timedelta(seconds=int(round(seconds)))\n    return t.strftime("%d/%m/%Y"), t.strftime("%H:%M:%S")\n\n\ndef details_rows(airings: list[Airing], index: LibraryIndex, name: BroadcastName) -> list[dict]:\n    rows = []\n    for k, a in enumerate(airings, 1):\n        day, start = clock(name, a.start_s)\n        _, end = clock(name, a.end_s)\n        rows.append(dict(zip(DETAIL_COLUMNS, [\n            k, day, start, end, round(a.aired_s, 1), a.name,\n            round(index.spots[a.spot_id].duration_s, 1), int(round(100 * a.coverage)),\n            a.status, ", ".join(a.also_matched), hms(a.start_s)], strict=True)))\n    return rows\n\n\ndef summary_rows(airings: list[Airing], index: LibraryIndex, name: BroadcastName) -> list[dict]:\n    """One row per library spot (0 airings included), most aired first."""\n    rows = []\n    for sid, spot in enumerate(index.spots):\n        mine = [a for a in airings if a.spot_id == sid]\n        rows.append(dict(zip(SUMMARY_COLUMNS, [\n            spot.name, len(mine), ", ".join(clock(name, a.start_s)[1] for a in mine),\n            sum(a.status.startswith("Phát thiếu") for a in mine),\n            sum(a.status.startswith("Cần nghe lại") for a in mine)], strict=True)))\n    return sorted(rows, key=lambda r: (-r["Số lần phát"], r["Tên spot"]))\n\n\ndef _sheet(wb: Workbook, title: str, columns: list[str], rows: list[dict]) -> None:\n    ws = wb.create_sheet(title)\n    ws.append(columns)\n    for r in rows:\n        ws.append([r[c] for c in columns])\n        for cell in ws[ws.max_row]:\n            if isinstance(cell.value, str) and cell.value.startswith("="):\n                cell.data_type = "s"  # a file named \'=...\' stays text, never a formula\n    for cell in ws[1]:\n        cell.font = Font(bold=True)\n    ws.freeze_panes = "A2"\n    for col in ws.columns:\n        width = max(len(str(c.value)) for c in col if c.value is not None)\n        ws.column_dimensions[col[0].column_letter].width = min(60, width + 2)\n\n\ndef write_report(path: Path, details: list[dict], summary: list[dict],\n                 info: list[tuple[str, str]]) -> None:\n    """Write the three sheets atomically, so a half-written report never appears."""\n    path.parent.mkdir(parents=True, exist_ok=True)\n    wb = Workbook()\n    wb.remove(wb.active)\n    _sheet(wb, "Chi tiết", DETAIL_COLUMNS, details)\n    _sheet(wb, "Tổng hợp", SUMMARY_COLUMNS, summary)\n    _sheet(wb, "Thông tin", ["Mục", "Giá trị"], [{"Mục": k, "Giá trị": v} for k, v in info])\n    tmp = path.with_name(path.stem + ".tmp.xlsx")\n    wb.save(tmp)\n    os.replace(tmp, path)\n', 'radioqc/ui.py': '"""Tiny HTML helpers for the notebook, so its cells need neither pandas nor anything else."""\nfrom __future__ import annotations\n\nimport html\nfrom collections.abc import Iterable, Sequence\n\n_ALERT = {"LỖI": "color:#c00;font-weight:bold", "CẢNH BÁO": "color:#b60", "Cần nghe lại": "color:#b60",\n          "CHỜ FILE": "color:#1a73e8"}\n\n\ndef _style(value: str) -> str:\n    return next((css for prefix, css in _ALERT.items() if value.startswith(prefix)), "")\n\n\ndef html_table(headers: Sequence[str], rows: Iterable[Sequence[object]]) -> str:\n    """An escaped HTML table; error, warning and waiting cells are coloured."""\n    def cell(tag: str, value: object) -> str:\n        text = str(value)\n        return f\'<{tag} style="text-align:left;{_style(text)}">{html.escape(text)}</{tag}>\'\n\n    head = "<tr>" + "".join(cell("th", h) for h in headers) + "</tr>"\n    body = "".join("<tr>" + "".join(cell("td", v) for v in row) + "</tr>" for row in rows)\n    return f"<table>{head}{body}</table>"\n\n\ndef html_note(text: str) -> str:\n    """A highlighted \'what to do next\' box, readable in light and dark themes."""\n    return (\'<div style="margin:8px 0;padding:10px 14px;border-left:4px solid #1a73e8;\'\n            f\'background:#e8f0fe;color:#174ea6;font-size:14px">{html.escape(text)}</div>\')\n'}
_base = pathlib.Path("/content/radioqc_code")
for _rel, _src in _FILES.items():
    _p = _base / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_text(_src, encoding="utf-8")
if str(_base) not in sys.path:
    sys.path.insert(0, str(_base))
for _m in [m for m in sys.modules if m == "radioqc" or m.startswith("radioqc.")]:
    del sys.modules[_m]  # pick up new code when this cell is re-run
from google.colab import drive
drive.mount("/content/drive")
from IPython.display import HTML, display
from radioqc import __version__
from radioqc.pipeline import Layout, next_step, self_check
from radioqc.ui import html_note, html_table
layout = Layout(pathlib.Path("/content/drive/MyDrive") / THU_MUC)
layout.ensure()
print(f"radioqc {__version__} | thư mục: {layout.root}")
display(HTML(html_table(["Mục", "Trạng thái", "Chi tiết"], self_check(layout))))
_next = next_step(layout)
if _next:
    display(HTML(html_note(_next)))


In [ ]:
#@title 2️⃣ Chạy chế độ A – Đối soát
from radioqc.pipeline import run_mode_a
_res = run_mode_a(layout, rerun=CHAY_LAI_TAT_CA)
if _res:
    display(HTML(html_table(["File", "Trạng thái", "Số lần phát", "Báo cáo (3_KetQua)", "Thời gian (s)"],
                            [(r.file, r.status, r.airings, r.report, round(r.seconds, 1)) for r in _res])))


In [ ]:
#@title 3️⃣ Nghe lại một lần phát
FILE = ""  #@param {type:"string"}
STT = 1  #@param {type:"integer"}
from IPython.display import Audio
from radioqc.pipeline import listen_clip
try:
    _x, _sr, _label = listen_clip(layout, FILE, STT)
except ValueError as _e:  # nothing to play yet, or a wrong STT: a hint, not a crash
    print(f"ℹ️ {_e}")
else:
    print(_label)
    display(Audio(_x, rate=_sr))
